# Train a Dataset Builder zip

Trains yolov8n-seg on this dataset and gives back `best.onnx`.

1. Set **Runtime → Change runtime type → GPU**.
2. Run the cells in order. The second asks for the zip: upload **your dataset zip**.

In [ ]:
!pip install -q ultralytics

In [ ]:
from google.colab import files
up = files.upload()  # pick your dataset zip
zip_name = next(iter(up))
!rm -rf /content/ds && mkdir -p /content/ds && unzip -q "{zip_name}" -d /content/ds
!cat /content/ds/data.yaml

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n-seg.pt")
model.train(data="/content/ds/data.yaml", imgsz=512, epochs=100, patience=20, batch=32, fliplr=0.0, mosaic=1.0, project="/content/runs", name="seg")

In [ ]:
best = YOLO("/content/runs/seg/weights/best.pt")
best.val(data="/content/ds/data.yaml", split="test", imgsz=512)
best.export(format="onnx", imgsz=512, opset=12, simplify=True)
files.download("/content/runs/seg/weights/best.onnx")

The ONNX keeps no class names: they are `names` in `data.yaml`, in that order.

A run that stopped resumes with `YOLO("/content/runs/seg/weights/last.pt").train(resume=True)`.